**Role in the paper.** Combines the per-city results (`results/data/<City>/UHI_<City>_annualcycle_method_comparison_tasmin.csv`) of the nine cities: JJA mean UHIN per method (observed, simulated, bias) and the mean over the cities (Turku excluded from the a2a mean, see SI). Writes the LaTeX tables to `results/tables/comparing_methods/` and the SI dot plots to `results/figures/Supplementary_Information/`.

| Code name (columns / variables) | Paper name |
|---|---|
| `p2p_un_ucdb` | p2p_UCN_uc |
| `p2p_un_lcz` | p2p_UCN_lcz |
| `p2p_ms_ucdb` | p2p_MS_uc |
| `urclim_un_ucdb` | a2a_UCN_uc |
| `urclim_un_lcz` | a2a_UCN_lcz |
| `urclim_ms_ucdb` | a2a_MS_uc |
| `blend_ms_ucdb` (method 2, blended) | p2a_MS_dist |

In [ ]:
import pandas as pd
from pathlib import Path
import xarray as xr
ROOT = Path.cwd().parent.parent.parent
import numpy as np

In [ ]:
var='tasmin'
season = 'JJA'

if var=='tasmin':
    suffix = 'N'
elif var=='tasmax':
    suffix = 'X'

if season=='JJA':
    months = [6,7,8]
if season=='DJF':
    months = [12,1,2]
if season == 'YEAR':
    months = [1,2,3,4,5,6,7,8,9,10,11,12]
    

In [ ]:
suffix = {'tasmin':'N', 'tasmax':'X'}[var]

dfs = {v:{} for v in ['bias','sim','obs']}
cities = [ 'Amsterdam', 'Barcelona','Berlin','Birmingham','Ghent', 'Lisbon', 'Rennes', 'Turku','Zurich' ]

for city in cities:
    df = pd.read_csv(ROOT/f"results/data/{city}/UHI_{city}_annualcycle_method_comparison_{var}.csv")
    for v in dfs:
        df = df[df['month'].isin(months)]
        s = df.filter(like=f"_{v}").mean()
        s.index = pd.MultiIndex.from_tuples([tuple(c.replace(f"UHI{suffix}_","").replace(f"_{v}","").split("_")) for c in s.index],
                                             names=["aggregation","network","classification"])
        dfs[v][city] = s

df_bias, df_sim, df_obs = [pd.DataFrame(x).T for x in dfs.values()]

#for each method, compute the mean bias, obs and sim across all cities with spread and store in a new row in df_bias
#exclude Turku form the mean calculation for aggregation = 'urclim' in df_bias and df_sim  
# Exclude Turku from the mean calculation for urclim only
exclude_turku = True
if exclude_turku:
    df_bias_mean = df_bias.copy()
    df_sim_mean = df_sim.copy()

    urclim_cols = df_bias.columns.get_level_values("aggregation") == "urclim"

    df_bias_mean.loc["Turku", urclim_cols] = np.nan
    df_sim_mean.loc["Turku", urclim_cols] = np.nan

    # Calculate means
    df_bias.loc["mean"] = df_bias_mean.mean()
    df_sim.loc["mean"] = df_sim_mean.mean()

    # Obs remains unchanged
    df_obs.loc["mean"] = df_obs.mean()

else:
    df_bias.loc["mean"] = df_bias.mean()
    df_sim.loc["mean"] = df_sim.mean()
    df_obs.loc["mean"] = df_obs.mean()

In [ ]:
major = [('p2p','ms','ucdb'),
         ('urclim','ms','ucdb')]

minor = [('p2p','un','lcz'),
         ('urclim','un','lcz')]
classes = pd.DataFrame('', index=df_bias.index, columns=df_bias.columns)

for c in major:
    classes[c] = 'major'

for c in minor:
    classes[c] = 'minor'

In [ ]:
cols = [
    ('p2p','un','ucdb'),
    ('p2p','un','lcz'),
    ('p2p','ms','ucdb'),
    ('urclim','un','ucdb'),
    ('urclim','un','lcz'),
    ('urclim','ms','ucdb'),
    ('blend','ms','ucdb')
]

df = df_bias[cols]

In [ ]:
from matplotlib import cm, colors



In [ ]:
def df_to_latex(df, caption="", label="", fmt=".2f", vmin=-2, vmax=2, cmap_name="coolwarm"):

    norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=np.mean([vmin,vmax]), vmax=vmax)
    cmap = cm.get_cmap(cmap_name)
    r0, g0, b0, _ = cmap(norm(0))
    r0, g0, b0 = int(r0*255), int(g0*255), int(b0*255)

    def latex_cell(value):
        if pd.isna(value):
            return rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}--"

        r, g, b, _ = cmap(norm(value))
        r, g, b = int(r*255), int(g*255), int(b*255)

        return rf"\cellcolor[RGB]{{{r},{g},{b}}}{value:.1f}"

    lines = []

    lines.append(r"\begin{table*}[t]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(r"\begin{tabular}{lccc ccc c}")
    lines.append(r"\toprule")

    lines.append(r"& \multicolumn{3}{c}{p2p} & \multicolumn{3}{c}{a2a} & p2a \\")
    lines.append(r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-8}")

    lines.append(r"& \multicolumn{2}{c}{UCN} & MS & \multicolumn{2}{c}{UCN} & MS & MS \\")
    lines.append(r"\cmidrule(lr){2-3}\cmidrule(lr){5-6}")

    lines.append(r"City & uc & lcz & uc & uc & lcz & uc & dist \\")
    lines.append(r"\midrule")

    for city, row in df.iterrows():
        if city == "mean":
            lines.append(r"\midrule")          # horizontal line above mean
        
        vals = " & ".join(latex_cell(v) for v in row)
        # if city = "Turku" and exclude_turku is True, then color the urclim columns in gray but print the values 
        if city == "Turku" and exclude_turku:
            print(f'{row[0]:.1f}')
            vals = vals.split(" & ")
            vals[3] =  rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}{row[3]:.1f}"
            vals[4] = rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}{row[4]:.1f}"
            vals[5] = rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}{row[5]:.1f}"
            vals = " & ".join(vals)
        lines.append(f"{city} & {vals} \\\\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")

    lines.append(r"\end{table*}")

    return "\n".join(lines)

In [ ]:
latex_bias = df_to_latex(
    df_bias,
    caption=f"Mean UHI{suffix} {season} bias (°C) for all cities and evaluation methods.",
    label="tab:bias",
     vmin=-2., vmax=2., 
)

with open(ROOT / "results/tables/comparing_methods/bias_table.tex","w") as f:
    f.write(latex_bias)

In [ ]:
from matplotlib import cm, colors

def df_obs_sim_to_latex(
    df_obs,
    df_sim,
    caption="",
    label="",
    cmap_name="YlOrRd",
    vmin=-1,
    vmax=3,
    vcenter=None
):

    cmap = cm.get_cmap(cmap_name)
    if vcenter is not None:
        # Use diverging normalization only if 0 is inside the range
        if vmin < 0 < vmax:
            norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        else:
            norm = colors.Normalize(vmin=vmin, vmax=vmax)
    else:
        norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)


    r0, g0, b0, _ = cmap(norm(0))
    r0, g0, b0 = int(r0*255), int(g0*255), int(b0*255)

    def latex_cell(value):
        if pd.isna(value):
            return rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}--"

        r, g, b, _ = cmap(norm(value))
        r, g, b = int(r*255), int(g*255), int(b*255)

        return rf"\cellcolor[RGB]{{{r},{g},{b}}}{value:.1f}"

    lines = []

    lines.append(r"\begin{table*}[t]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(
        r"\begin{tabular}{l ccc ccc c @{\hspace{1.2em}\vrule\hspace{1.2em}} ccc ccc c}"
    )

    lines.append(r"\toprule")

    lines.append(
        r"& \multicolumn{7}{c}{Observed}"
        r"& \multicolumn{7}{c}{Simulated}\\"
    )

    lines.append(r"\cmidrule(lr){2-8}\cmidrule(lr){9-15}")

    lines.append(
        r"& \multicolumn{3}{c}{p2p}"
        r"& \multicolumn{3}{c}{a2a}"
        r"& p2a"
        r"& \multicolumn{3}{c}{p2p}"
        r"& \multicolumn{3}{c}{a2a}"
        r"& p2a\\"
    )

    lines.append(
        r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}"
        r"\cmidrule(lr){9-11}\cmidrule(lr){12-14}"
    )

    lines.append(
        r"& \multicolumn{2}{c}{UCN} & MS"
        r"& \multicolumn{2}{c}{UCN} & MS"
        r"& MS"
        r"& \multicolumn{2}{c}{UCN} & MS"
        r"& \multicolumn{2}{c}{UCN} & MS"
        r"& MS\\"
    )
    lines.append(r"\cmidrule(lr){2-3}\cmidrule(lr){5-6}\cmidrule(lr){9-10}\cmidrule(lr){12-13}")

    lines.append(
        r"City"
        r"& uc & lcz & uc & uc & lcz & uc & dist"
        r"& uc & lcz & uc & uc & lcz & uc & dist\\"
    )

    lines.append(r"\midrule")

    for city in df_obs.index:

        if city == "mean":
            lines.append(r"\midrule")

        obs = " & ".join(latex_cell(v) for v in df_obs.loc[city])
        sim = " & ".join(latex_cell(v) for v in df_sim.loc[city])
        if city == "Turku" and exclude_turku:
            sim = sim.split(" & ")
            sim[3] = rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}" + f'{df_sim.loc[city][3]:.1f}' 
            sim[4] =  rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}" + f'{df_sim.loc[city][4]:.1f}'
            sim[5] =  rf"\cellcolor[RGB]{{{r0},{g0},{b0}}}"+ f'{df_sim.loc[city][5]:.1f}'
            sim = " & ".join(sim)

        lines.append(f"{city} & {obs} & {sim} \\\\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")

    lines.append(r"\end{table*}")

    return "\n".join(lines)

In [ ]:
latex = df_obs_sim_to_latex(
    df_obs,
    df_sim,
    caption=f"Mean {season} UHI{suffix} (°C) for all cities and evaluation methods: observed (left) and simulated (right).",
    label="tab:obs_sim",
    cmap_name="RdYlBu_r",
    vmin=-1,
    vcenter = 0,
    vmax=3.5,
)

with open(ROOT / "results/tables/comparing_methods/obs_sim_table.tex", "w") as f:
    f.write(latex)

In [ ]:
latex_obs = df_to_latex(
    df_obs,
    caption=f"{season} mean observed UHI (°C)",
    label="tab:obs",
    vmin=0, vmax=3, cmap_name="YlOrRd"
)

with open(ROOT / "results/tables/comparing_methods/obs_table.tex","w") as f:
    f.write(latex_obs)

latex_sim = df_to_latex(
    df_sim,
    caption=f"{season} mean simulated UHI (°C)",
    label="tab:sim",
    vmin=0, vmax=3, cmap_name="YlOrRd"   
)
with open(ROOT / "results/tables/comparing_methods/sim_table.tex","w") as f:
    f.write(latex_sim)
    

In [ ]:
sty = df_bias.style.background_gradient(cmap='coolwarm', axis=None, vmin =-1.5, vmax=1.5).format("{:.1f} °C").set_table_styles([{'selector':'th','props':[('text-align','center')]}])
sty.apply(lambda x: ['border-top: 2px solid black' if x.name == 'mean' else '' for i in x], axis=1)
sty

In [ ]:
latex = sty.to_latex(convert_css=True)
with open(ROOT / "results/tables/comparing_methods/table.tex", "w") as f:
    f.write(latex)


In [ ]:
df_bias.to_latex(float_format="%.1f")

In [ ]:
sty = df_obs.style.background_gradient(cmap='Reds', axis=None, vmax= 3).format("{:.1f} °C").set_table_styles([{'selector':'th','props':[('text-align','center')]}])
sty.apply(lambda x: ['border-top: 2px solid black' if x.name == 'mean' else '' for i in x], axis=1)
sty


In [ ]:
sty = df_sim.style.background_gradient(cmap='Reds', axis=None, vmax = 3).format("{:.1f} °C").set_table_styles([{'selector':'th','props':[('text-align','center')]}])
sty.apply(lambda x: ['border-top: 2px solid black' if x.name == 'mean' else '' for i in x], axis=1)

sty

## Dot plot per city: UHI bias per evaluation characteristic (SI figure)
Panels: (a)  UR grid cell selection p2p/a2a/p2a, (b) network UCN vs MS (uc), (c) classification uc vs lcz (UCN). Hollow markers = Turku a2a (30% urban-fraction threshold, excluded from the mean).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---- dot plot per city: one panel per evaluation characteristic (Fig. S-dotplot) ----
agg_lbl = {'p2p':'p2p', 'urclim':'a2a', 'blend':'p2a'}
net_lbl = {'un':'UCN', 'ms':'MS'}
cls_lbl = {'ucdb':'uc', 'lcz':'lcz'}

# Okabe-Ito colours (colour-blind safe); marker shape is the secondary encoding
panels = [
    dict(title=' UR grid cell selection', colour_key=0,
         colours={'p2p':'#0072B2', 'urclim':'#E69F00', 'blend':'#CC79A7'},
         marker_key=lambda c: (c[1], c[2] if c[1]=='un' else 'ucdb'),
         markers={('un','ucdb'):'o', ('un','lcz'):'s', ('ms','ucdb'):'^'},
         marker_lbl={('un','ucdb'):'UCN · uc', ('un','lcz'):'UCN · lcz', ('ms','ucdb'):'MS · uc / dist'},
         groups=[[('p2p','un','ucdb'), ('urclim','un','ucdb')],
                 [('p2p','un','lcz'),  ('urclim','un','lcz')],
                 [('p2p','ms','ucdb'), ('urclim','ms','ucdb'), ('blend','ms','ucdb')]]),
    dict(title='Observational network (uc)', colour_key=1,
         colours={'un':'#009E73', 'ms':'#D55E00'},
         marker_key=lambda c: c[0],
         markers={'p2p':'o', 'urclim':'s'},
         marker_lbl={'p2p':'p2p', 'urclim':'a2a'},
         groups=[[('p2p','un','ucdb'),    ('p2p','ms','ucdb')],
                 [('urclim','un','ucdb'), ('urclim','ms','ucdb')]]),
    dict(title='Station classification (UCN)', colour_key=2,
         colours={'ucdb':'#56B4E9', 'lcz':'#000000'},
         marker_key=lambda c: c[0],
         markers={'p2p':'o', 'urclim':'s'},
         marker_lbl={'p2p':'p2p', 'urclim':'a2a'},
         groups=[[('p2p','un','ucdb'),    ('p2p','un','lcz')],
                 [('urclim','un','ucdb'), ('urclim','un','lcz')]]),
]
colour_lbl = [agg_lbl, net_lbl, cls_lbl]


def plot_method_dots(df, ylabel, fname, hollow_turku_a2a=True, show_mean=True):
    """df: cities (+ 'mean') x MultiIndex(aggregation, network, classification)."""
    city_list = [c for c in df.index if c != 'mean']
    xs = {c: i for i, c in enumerate(city_list)}
    if show_mean:
        xs['mean'] = len(city_list) + 0.4

    fig, axes = plt.subplots(3, 1, figsize=(24/2.54, 20/2.54), sharex=True)
    for ax, p, letter, clbl in zip(axes, panels, 'abc', colour_lbl):
        # fixed (non-random) horizontal offsets: pairs side by side, small gap between groups
        n_tot = sum(len(g) for g in p['groups'])
        step, gap = 0.09, 0.05
        width = step*(n_tot - 1) + gap*(len(p['groups']) - 1)
        offs, o = [], -width/2
        for g in p['groups']:
            offs.append([o + step*k for k in range(len(g))])
            o = offs[-1][-1] + step + gap

        for city, x0 in xs.items():
            for g, og in zip(p['groups'], offs):
                xg = [x0 + d for d in og]
                yg = [df.loc[city, c] for c in g]
                ax.plot(xg, yg, color='0.6', lw=0.8, zorder=1)
                for xx, yy, c in zip(xg, yg, g):
                    if np.isnan(yy):
                        continue
                    col = p['colours'][c[p['colour_key']]]
                    hollow = hollow_turku_a2a and city == 'Turku' and c[0] == 'urclim'
                    ax.scatter(xx, yy, s=28, marker=p['markers'][p['marker_key'](c)],
                               facecolor='white' if hollow else col, edgecolor=col,
                               linewidth=1.2, zorder=3)

        ax.axhline(0, color='k', ls=':', lw=1, zorder=0)
        # faint separators between cities (lighter/looser dots than the zero line)
        for xv in np.arange(len(city_list) - 1) + 0.5:
            ax.axvline(xv, color='0.75', ls=(0, (1, 3)), lw=0.7, zorder=0)
        if show_mean:
            ax.axvline(len(city_list) - 0.3, color='0.3', lw=0.8)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.tick_params(labelsize=11)
        ax.set_title(f'({letter}) {p["title"]}', loc='left', fontsize=12)
        for s in ['top', 'right']:
            ax.spines[s].set_visible(False)

        handles = [Line2D([], [], ls='', marker='o', ms=7, color=v, label=clbl[k])
                   for k, v in p['colours'].items()]
        handles += [Line2D([], [], ls='', marker=m, ms=7, mfc='0.5', mec='0.5', label=p['marker_lbl'][k])
                    for k, m in p['markers'].items()]
        if hollow_turku_a2a:
            handles.append(Line2D([], [], ls='', marker='o', ms=7, mfc='white', mec='0.5',
                                  label='Turku a2a\n(30% threshold)'))
        ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.0, 1.0),
                  fontsize=10, frameon=False, handletextpad=0.2)

    # common y-limits so the panels can be compared
    lo = min(a.get_ylim()[0] for a in axes); hi = max(a.get_ylim()[1] for a in axes)
    for a in axes:
        a.set_ylim(lo, hi)
    axes[-1].set_xticks(list(xs.values()))
    axes[-1].set_xticklabels([c if c != 'mean' else 'Mean' for c in xs], rotation=45, ha='right', fontsize=11)
    fig.tight_layout()
    out = ROOT / 'results/figures/Supplementary_Information'
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / fname, bbox_inches='tight')
    return fig


plot_method_dots(df_bias, f'{season} UHI{suffix} bias (°C)', f'dotplot_UHI{suffix}_bias_{season}_per_method.pdf')
plot_method_dots(df_obs, f'Observed {season} UHI{suffix} (°C)', f'dotplot_UHI{suffix}_obs_{season}_per_method.pdf',
                 hollow_turku_a2a=False)


## Older exploratory cells (not used in the paper)

In [ ]:
#make two similar d', f's one for months DJF (12,1,2 in biases.month) and one for JJA
df_djf = pd.DataFrame(columns=['City', ', f'UHI{suffix}_p2p_un_bias', ', f'UHI{suffix}_p2p_ms_bias' , ', f'UHI{suffix}_blend_bias', ', f'UHI{suffix}_urclim_un_bias', ', f'UHI{suffix}_urclim_ms_bias'])
for city in [  'Amsterdam','Berlin','Birmingham','Rennes','Turku', 'Zurich', 'Ghent', 'Barcelona']:
    save_dir =  ROOT / ', f'results/data/{city}/'
    biases = xr.open_dataset(f"{save_dir}/UHIN_{city}_annualcycle_method_comparison_biases_{var}.nc")
    biases= biases.sel(month=[12,1,2])
    df_djf.loc[len(df_djf)] = [city, biases[', f'UHI{suffix}_p2p_un_bias'].mean().item(), biases[', f'UHI{suffix}_p2p_ms_bias'].mean().item() , biases[', f'UHI{suffix}_blend_bias'].mean().item(), biases[', f'UHI{suffix}_urclim_un_bias'].mean().item(), biases[', f'UHI{suffix}_urclim_ms_bias'].mean().item()]
 
df_djf.set_index('City', inplace=True)
df_djf.style.background_gradient(cmap='coolwarm', axis=None, vmin=-2, vmax=2).format("{:.2f} °C")

In [ ]:
#make two similar d', f's one for months DJF (12,1,2 in biases.month) and one for JJA
df_jja = pd.DataFrame(columns=['City', ', f'UHI{suffix}_p2p_un_bias', ', f'UHI{suffix}_p2p_ms_bias' , ', f'UHI{suffix}_blend_bias', ', f'UHI{suffix}_urclim_un_bias', ', f'UHI{suffix}_urclim_ms_bias'])
for city in [  'Amsterdam','Berlin','Birmingham','Rennes','Turku', 'Zurich', 'Ghent', 'Barcelona']:
    save_dir =  ROOT / ', f'results/data/{city}/'
    biases = xr.open_dataset(f"{save_dir}/UHIN_{city}_annualcycle_method_comparison_biases_{var}.nc")
    biases= biases.sel(month=[6,7,8])
    df_jja.loc[len(df_jja)] = [city, biases[', f'UHI{suffix}_p2p_un_bias'].mean().item(), biases[', f'UHI{suffix}_p2p_ms_bias'].mean().item() , biases[', f'UHI{suffix}_blend_bias'].mean().item(), biases[', f'UHI{suffix}_urclim_un_bias'].mean().item(), biases[', f'UHI{suffix}_urclim_ms_bias'].mean().item()]
 
df_jja.set_index('City', inplace=True)
df_jja.style.background_gradient(cmap='coolwarm', axis=None, vmin=-2, vmax=2).format("{:.2f} °C")
